In [ ]:
# Practice Problems Day 17

### Author: Joanna Bieri (solutions)

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### Day 17 Practice - SOLUTIONS
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**These are practice problems. They are not graded and you do not turn them in.** They are here so you can practice today's code: one transformer block written as a class, and a review classifier built around it. Plan on about 45 minutes. Each training run takes a few seconds on a laptop.

**These are my solutions.** Try every problem yourself first, then come here to check your work. There is more than one right way to write the code, and your words will be different from mine, that is fine! If your numbers do not match, look for the difference before you look at my code.

**Reading:** Geron chapter 15, **Positional encodings**, **Multi-Head Attention (MHA)** and **Building the rest of the Transformer**.

GOALS:

1. Write a transformer block as a class: multi-head attention, skip, layer norm, feed-forward.
2. Use the block on the long reviews with one feeling word, where the RNN failed.
3. Explain in words what positions and the other pieces of the block are for.

**How much of this should you write yourself?** The data cell is given, with the training function from the notes. Problem 1 gives you most of the block, you write the missing lines. Problem 2 lists the steps. Problem 3 is words only. The **Optional exploration** at the end is extra, only if you are curious.

**Save your work.** Commit and push when you are done, so you have your code for later:

```bash
git add .
git commit -m "Day 17 practice"
git push
```

---

In [ ]:
# RUN THIS. The words, both review makers and the training function from the notes.
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

feeling_words = ["great", "good", "fun", "loved", "wonderful",       # positive, ids 0 to 4
                 "bad", "boring", "awful", "hated", "terrible"]      # negative, ids 5 to 9
filler_words = ["the", "movie", "was", "and", "a", "plot", "actors", "it", "story", "long",
                "music", "ending", "i", "saw", "with", "my", "friends", "on", "friday", "popcorn"]
vocab = feeling_words + filler_words
vocab_size = len(vocab)

# One feeling word reviews (like the RNN notes): ONE feeling word at a random spot decides
def make_reviews_one(n_reviews, n_words, seed):
    generator = torch.Generator().manual_seed(seed)
    X = torch.randint(10, vocab_size, (n_reviews, n_words), generator=generator)
    feeling = torch.randint(0, 10, (n_reviews,), generator=generator)
    spot = torch.randint(0, n_words, (n_reviews,), generator=generator)
    X[torch.arange(n_reviews), spot] = feeling
    y = (feeling < 5).long()
    return X, y

# Today's reviews: TWO feeling words, the LAST one decides
def make_reviews_two(n_reviews, n_words, seed):
    generator = torch.Generator().manual_seed(seed)
    X = torch.randint(10, vocab_size, (n_reviews, n_words), generator=generator)
    positive = torch.randint(0, 5, (n_reviews,), generator=generator)
    negative = torch.randint(5, 10, (n_reviews,), generator=generator)
    spot_pos = torch.randint(0, n_words, (n_reviews,), generator=generator)
    spot_neg = (spot_pos + torch.randint(1, n_words, (n_reviews,), generator=generator)) % n_words
    X[torch.arange(n_reviews), spot_pos] = positive
    X[torch.arange(n_reviews), spot_neg] = negative
    y = (spot_pos > spot_neg).long()
    return X, y

def train_model(model, X_train, y_train, X_valid, y_valid, n_epochs):
    torch.manual_seed(42)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    valid_accuracy = []
    for epoch in range(n_epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = loss_fn(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            y_pred = model(X_valid).argmax(dim=1)
            accuracy = (y_pred == y_valid).float().mean().item()
        valid_accuracy.append(accuracy)
        print("epoch", epoch + 1, "  valid accuracy:", round(accuracy, 4))
    return valid_accuracy

print("vocabulary size:", vocab_size)

## Problem 1: Write the transformer block

**What to copy, what to write:** Most of the block is given. Write the lines the comments ask for. The notes have the same class, but try to write it without looking first. The picture of the block in the notes is a good guide.

**1a.** Fill in `forward` below. Then send 8 made up reviews of 10 words, `torch.randn(8, 10, 16)`, through a new block and print the shape that comes out.

**1b.** Count the numbers your block has to learn with `n_heads=2`. Then make a block with `n_heads=4` and count again. Did the count change? Use the picture of multi-head attention to explain why.

In [ ]:
class TransformerBlock(nn.Module):

    def __init__(self, n_numbers=16, n_heads=2):
        super().__init__()
        self.attention = nn.MultiheadAttention(n_numbers, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(n_numbers)
        self.feed_forward = nn.Sequential(
            nn.Linear(n_numbers, 2 * n_numbers),
            nn.ReLU(),
            nn.Linear(2 * n_numbers, n_numbers))
        self.norm2 = nn.LayerNorm(n_numbers)

    def forward(self, X):
        attention_out, weights = self.attention(X, X, X)     # TWO things come back
        Z = self.norm1(X + attention_out)                     # skip, then layer norm
        out = self.norm2(Z + self.feed_forward(Z))            # feed-forward, skip, layer norm
        return out

torch.manual_seed(42)
block = TransformerBlock()
X_fake = torch.randn(8, 10, 16)
print("coming out:", block(X_fake).shape)

In [ ]:
# 1b: count the numbers to learn with 2 heads, then with 4 heads
n_2_heads = 0
for p in TransformerBlock(16, n_heads=2).parameters():
    n_2_heads = n_2_heads + p.numel()

n_4_heads = 0
for p in TransformerBlock(16, n_heads=4).parameters():
    n_4_heads = n_4_heads + p.numel()

print("2 heads:", n_2_heads)
print("4 heads:", n_4_heads)

**1a.** The three missing lines are `attention_out, weights = self.attention(X, X, X)`, `Z = self.norm1(X + attention_out)` and `out = self.norm2(Z + self.feed_forward(Z))`. A block takes in (8, 10, 16) and gives back `torch.Size([8, 10, 16])`, the same shape. That is what lets us stack blocks.

**1b.** 2,224 numbers with 2 heads and 2,224 with 4 heads. The count does not change. The heads split the same 16 numbers between them (2 heads of 8, or 4 heads of 4), so the query, key, value and output layers are the same size either way. More heads only means each head works with fewer numbers.

---

## Problem 2: The long reviews with one feeling word (steps outlined)

In the RNN notes a plain RNN got about 0.5 on 100 word reviews where **one** feeling word, anywhere in the review, decides. The GRU learned them after a few epochs. Now let's try your transformer block!

**What to copy, what to write:** Copy the `ReviewTransformer` class from the notes (it uses your `TransformerBlock`). Write the steps below yourself.

**2a.** Make the reviews with one feeling word, 100 words long, with `make_reviews_one` (4,000 to train on with `seed=0`, 1,000 to validate on with `seed=1`). Train a `ReviewTransformer` **without** positions on them for 5 epochs.

**2b.** How does it compare to the RNN (0.493 after 5 epochs)? Give two reasons it can learn these reviews: one about **positions** and one about how far the gradient has to travel back to the feeling word.

In [ ]:
# Step 1: the ReviewTransformer class from the notes
class ReviewTransformer(nn.Module):

    def __init__(self, n_words, use_positions=True):
        super().__init__()
        self.use_positions = use_positions
        self.embed = nn.Embedding(vocab_size, 16)
        self.position_embed = nn.Embedding(n_words, 16)
        self.block = TransformerBlock(16, n_heads=2)
        self.out = nn.Linear(16, 2)

    def forward(self, X):
        vectors = self.embed(X)
        if self.use_positions:
            spots = torch.arange(X.shape[1])
            vectors = vectors + self.position_embed(spots)
        Z = self.block(vectors)
        summary = Z.mean(dim=1)
        scores = self.out(summary)
        return scores

In [ ]:
# Step 2: the 100 word reviews with one feeling word
X_train_100, y_train_100 = make_reviews_one(4000, n_words=100, seed=0)
X_valid_100, y_valid_100 = make_reviews_one(1000, n_words=100, seed=1)

# Steps 3 and 4: no positions, 5 epochs
torch.manual_seed(42)
model_100 = ReviewTransformer(n_words=100, use_positions=False)
accuracy_100 = train_model(model_100, X_train_100, y_train_100, X_valid_100, y_valid_100, n_epochs=5)

**2a.** 1.0 after the first epoch, and it stays at 1.0 for all 5 epochs.

**2b.** The RNN ended at 0.493 on these reviews, no better than a coin. The transformer gets every one right. (1) Positions do not matter here: only WHICH feeling word is in the review decides, not where it is, so a model that can only see which words are there is enough. (2) Attention lets the output look at every word directly, so the feeling word is one step away no matter where it is. The gradient does not have to go back through 100 steps like it did in the RNN, so it does not vanish.

---

## Problem 3: In words

**3a.** "dog bites man" and "man bites dog" use the same three words. Explain to a friend in DATA 101 why a transformer block with no positions would give the same answer for both, and what the positional embedding adds to fix it.

**3b.** A friend says "a transformer reads the review one word at a time, like an RNN." What would you tell them? Use the words **attention** and **hidden state**.

**3a.** Your answer will be different, here is one way to say it:

Attention compares every word to every other word, and nothing in that comparison says where a word is. "dog bites man" and "man bites dog" have the same three words, so the block makes the same three outputs, just in a different order, and the average over the words is the same. The positional embedding adds a learned list of numbers for each spot, so "dog" in spot 0 and "dog" in spot 2 go in as different numbers.

**3b.** No, it sees every word at the same time. An RNN reads one word, updates its hidden state, then reads the next, so everything has to pass through that one memory. In a transformer block, attention lets every word look directly at every other word, so there is no hidden state carried from word to word. That is also why it can be trained faster: all the words are worked on at once.

---

## Optional exploration

Only if you are curious. Nothing here is needed later.

**A.** Make the two feeling word reviews from the notes (`make_reviews_two`, 20 words, `seed=0` and `seed=1`) and train a `ReviewTransformer` with positions, but with `n_heads=4` in the block. Compare to the notes (0.956 after 10 epochs with 2 heads). What happens if you try `n_heads=3`?

In [ ]:
# A: 4 heads on the two feeling word reviews
X_train_20, y_train_20 = make_reviews_two(4000, n_words=20, seed=0)
X_valid_20, y_valid_20 = make_reviews_two(1000, n_words=20, seed=1)

torch.manual_seed(42)
model_4 = ReviewTransformer(n_words=20, use_positions=True)
model_4.block = TransformerBlock(16, n_heads=4)       # swap in a block with 4 heads
accuracy_4 = train_model(model_4, X_train_20, y_train_20, X_valid_20, y_valid_20, n_epochs=10)

In [ ]:
# A: 3 heads does not work, because 16 numbers do not split evenly into 3 heads
try:
    TransformerBlock(16, n_heads=3)
except AssertionError as error:
    print("Error:", error)

**A.** With 4 heads it gets to 0.983 by epoch 3 and ends at 0.994 after 10 epochs, a bit better than 0.956 with 2 heads on this run. Each head now works with only 4 numbers. With 3 heads PyTorch stops with `embed_dim must be divisible by num_heads`: 16 numbers do not split evenly into 3 heads.